# **Learned features in CNNs: Practice**

Welcome to the practical part of the first part of the module [“Analysis of CNN-based models”](https://ai-interpretability.school/learn/9/0/0)!

Let us repeat the theses we have covered:

1. Convolutional neural networks extract the patterns of an image thanks to skip-connections.
2. The structures extracted from an image by a trained convolutional network can be seen by passing an example through the convolutional layers with their weights one after another.
3. Convolutional neural networks are able to extract human-understandable concepts from data.

To make sure of each of them, let us move on to the practice.

**Task 1. Which of the theses 1–3 proposed at the beginning contains a mistake?**

In [ ]:
!pip install torch torchvision -q


In [ ]:
import torch
import requests
import numpy as np
from tqdm import tqdm
from io import BytesIO
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
import matplotlib.pyplot as plt

We will work with ResNet18, also known as the Residual Network — a rather popular architecture for the classification task. The main feature of ResNet is the use of residual connections (skip connections), which makes it possible to train very deep networks efficiently, avoiding the problem of vanishing gradients.

In ResNet18, as the name says, 18 layers are used. The size of the input layer of the network is 224 x 224. After, roughly speaking, the image is taken into the network, convolutional layers with MaxPooling, Batch Normalization, and activation functions are applied one after another.

## The ResNet18 architecture

Picture 1.
![ResNet-18](https://raw.githubusercontent.com/SadSabrina/open-xai-materials/main/data/resnet18_architecture.png)

Source: [Dive into Deep Learning](https://d2l.ai/chapter_convolutional-modern/resnet.html) (Zhang, Lipton, Li, and Smola), [Wikimedia Commons](https://commons.wikimedia.org/wiki/File:Resnet-18_architecture_(rotated).svg), CC BY-SA 4.0.

![ResNet-18 blocks](https://raw.githubusercontent.com/SadSabrina/open-xai-materials/main/data/resnet18_basic_blocks.png)
Picture 2.

Let us load the pretrained model from PyTorch and look at its architecture.

In [ ]:
# Loading the model
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

#Examining the architecture
model

**Task 2. Which activation function is used in ResNet18?**  
**Task 3. How many convolutional layers does this ResNet18 architecture contain?**

Let us recall a few more details from the module:

- The first convolutional layers (counting from the input layer) learn very small components: edges and color transitions.
- The following layers learn separate textures.
- Closer to the last convolutional layers, patterns can be seen in the extracted information.
- At the end you can see images.

We also stated that these concepts can be seen in any trained convolutional neural network, by applying the convolutional layers with their weights to the data one after another.

Then let us formulate what we need:

1. To consider a trained network $net(img)$ containing $n$ convolutional layers with fixed weights $w_i, i=\vec{1, n}$.
2. For each layer inside the network:
    - if it is a convolutional one, extract the layer with its parameters;
    - otherwise move on.

To solve the task we could write a simple loop, but the ResNet18 architecture assumes that the convolutions are parts of a sequential (*Sequential*) layer. Taking this into account, our algorithm has to be transformed a little:

1. To consider a trained network $net(img)$ containing $n$ convolutional layers with fixed weights $w_i, i=\vec{1, n}$.
2. For each layer inside the network:
    - if it is a convolutional one, extract the layer with its parameters;
    - otherwise, if the layer is a Sequential one:
        - consider each of its components;
        - if the component is a convolutional layer, extract it with its parameters;
        - otherwise move on.

And this is the way we have to walk through the network. Let us do it below in the code.

In reality, for the visualization we will not extract all the convolutional layers: we will skip the convolutions in the *downsample* sub-blocks.

![temp-Imagermy-OBM.avif](https://raw.githubusercontent.com/SadSabrina/open-xai-materials/main/assets/temp-Imagermy-OBM.avif)

 The `print` operator does not reflect this, but the convolutions in downsample are applied not linearly, but in parallel with the convolutions above in the basic block (`BasicBlock`). Their output is added to the output of the main branch of the block and goes on into the next block, so by not extracting them separately we do not lose their contribution to the later maps.

In [ ]:
# Lists for extracting the convolutional layers and their weights
conv_weights = [] # list for the weights of a layer
conv_layers = [] # list for storing the layers themselves

counter = 0 # counter to make sure that we have extracted all the layers

# let us save all the components of the model into a list
model_children = list(model.children())


for i in range(len(model_children)):
    if type(model_children[i]) == nn.Conv2d:
        counter+=1
        conv_weights.append(model_children[i].weight)
        conv_layers.append(model_children[i])
    elif type(model_children[i]) == nn.Sequential:
      for j in range(0, len(model_children[i])):
        for child in model_children[i][j].children():
          if type(child) == nn.Conv2d:
                    counter+=1
                    conv_weights.append(child.weight)
                    conv_layers.append(child)
          elif type(child) == nn.Sequential and len(list(child.children())) != 0:
            for k in range(0, len(list(child.children()))):
              if type(list(child.children())[k]) == nn.Conv2d:
                counter+=1
                conv_weights.append(list(child.children())[k].weight)
                conv_layers.append(list(child.children())[k])


print(f'Layers extracted: {counter}')

**Task 4.** Fix the code below so that it extracts all the convolutional layers except those contained in the downsample sub-blocks. How many layers did you get after the fix?

In [ ]:
# Task 4: the loop below takes all convolutions of the network, like the cell above — 20 layers.
# Fix the condition so that the convolutions of the downsample sub-blocks do not get into the list
# (the name of such a module contains the word 'downsample').
conv_layers = []  # list for storing the layers themselves
for name, module in model.named_modules():
    if isinstance(module, nn.Conv2d):  # <- fix the condition
        conv_layers.append(module)
conv_weights = [layer.weight for layer in conv_layers]  # list for the weights of the layers
counter = len(conv_layers)

print(f'Layers extracted: {counter}')

**The input image.**

Let us load a specific example $x_0$.

Piggy photo: [Brett Sayles, Pexels](https://www.pexels.com/photo/4966539/), Pexels License.

In [ ]:
# Loading the image
url = 'https://raw.githubusercontent.com/SadSabrina/open-xai-materials/main/data/piglet.jpg'


image_bytes = requests.get(url).content
image = Image.open(BytesIO(image_bytes))
plt.imshow(image)
plt.show()

In [ ]:
print(f"Original image size: {image.size}")

In [ ]:
np.array(image).shape

Pillow opens a picture in the mode it was saved in: a jpg usually has three RGB channels, while a png often has four, RGBA, where the fourth one is responsible for the transparency of the pixel. The network expects exactly three channels, so convert the image to RGB explicitly: then the code works with a picture of any format.

In [ ]:
#Your code here

An image of any resolution can be fed to the input of ResNet18. Going through the convolutions one after another and thanks to `AdaptiveAvgPool2d` the image will be processed in any case. However, in order to get larger feature maps that are easy to look at, we will stretch the photo to 1224x1224 and normalize it.

Note that we set mean and std right away, without computing them. For models trained on ImageNet these values are common practice. They were computed on the basis of the images of the dataset.

In [ ]:
#Means
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

transform = transforms.Compose([
    transforms.Resize((1224, 1224)),  # We change the size of the image
    transforms.ToTensor(),  # We convert the picture into a PyTorch tensor
    transforms.Normalize(mean=mean, std=std)  # We normalize the picture
])

image = transform(image)
print(f"Image shape after resizing: {image.shape}")

In [ ]:
# Let us move the model to the GPU if that is possible
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = model.to(device)
image = image.to(device)

# Let us define the lists for extracting the feature maps
feature_maps = []  # List for the feature maps
layer_names = [str(number+1)+'_'+str(layer) for number, layer in enumerate(conv_layers)]  # List for the layer names

# We hang a hook on every convolution in conv_layers: during an ordinary forward pass of the network
# it stores the output of the convolution. This way the maps come from the actual computation of ResNet18,
# together with BatchNorm, ReLU, pooling, and the residual connections between the convolutions,
# and not from a chain of bare convolutions applied to one another.
hooks = [layer.register_forward_hook(lambda module, inp, out: feature_maps.append(out.detach()))
         for layer in conv_layers]
model.eval()
with torch.no_grad():
    model(image.unsqueeze(0))  # forward pass over a batch of one picture
for hook in hooks:
    hook.remove()
print(f'Feature maps: {len(feature_maps)}')

In [ ]:
# Preprocessing the feature maps into a form that can be visualized
processed_feature_maps = [] # List for storing the activation maps we get
for feature_map in feature_maps:
	feature_map = feature_map.squeeze(0) # We remove the batch dimension
	mean_feature_map = torch.sum(feature_map, 0) / feature_map.shape[0] # We compute the mean over the channels
	processed_feature_maps.append(mean_feature_map.data.cpu().numpy())

Finally, let us display the results we got.

In [ ]:
# Let us display the feature maps
fig = plt.figure(figsize=(30, 30))
for i in range(len(processed_feature_maps)):
	ax = fig.add_subplot(5, 4, i + 1)
	ax.imshow(processed_feature_maps[i])
	ax.axis("off")
	ax.set_title(layer_names[i].split('(')[0], fontsize=25)

**Task 5. Why did the piggy come out green?**

**Task 6. Experiment with `transforms.Resize` in `transform`. What changes as a result?**